# 🔍 POD Inference — Base vs Fine-Tuned (Unsloth / Official HF)

Toggle between:
- **Base model** vs **Fine-tuned model** (LoRA adapter)
- **Unsloth** vs **Official HuggingFace** pipeline

Dataset from **any** Kaggle account — auto-downloads via API if not attached.

---
## ⚙️ Config — Edit This Cell

In [ ]:
# ===================== CONFIGURATION =====================

# Toggle: "unsloth" or "official"
USE_PIPELINE = "unsloth"

# Toggle: True = load fine-tuned LoRA adapter, False = base model only
USE_FINETUNED = True

# Kaggle dataset — paste ANY URL (works for any account)
# If not attached as input, it will auto-download via Kaggle API
KAGGLE_DATASET_URL = "https://www.kaggle.com/datasets/puvithk/pod-clasification-v2"

# LoRA adapter path (HuggingFace repo or local path)
LORA_PATH = "puvith/qwen3-8b-pod-analyzer-v3"

# Base model IDs
OFFICIAL_BASE_MODEL = "Qwen/Qwen3-VL-8B-Instruct"
UNSLOTH_BASE_MODEL = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"

# Image resolution
MIN_PIXELS = 128 * 128
MAX_PIXELS = 512 * 512

# Number of eval samples (set 0 to skip eval)
NUM_EVAL_SAMPLES = 5

# =========================================================

import re, os
_match = re.search(r"kaggle\.com/datasets/(.+?)/?$", KAGGLE_DATASET_URL)
DATASET_SLUG = _match.group(1) if _match else None
DATASET_NAME = DATASET_SLUG.split("/")[-1] if DATASET_SLUG else "kaggle-dataset-upload"

# Check common Kaggle input paths
_candidates = [
    f"/kaggle/input/datasets/{DATASET_SLUG}",
    f"/kaggle/input/{DATASET_NAME}",
    f"/kaggle/working/{DATASET_NAME}",
]
DATASET_DIR = next((p for p in _candidates if os.path.exists(p)), None)

if DATASET_DIR:
    print(f"✅ Dataset found at: {DATASET_DIR}")
else:
    DATASET_DIR = f"/kaggle/working/{DATASET_NAME}"
    print(f"⚠️ Dataset not found locally — will download via API next cell")

print(f"Pipeline:     {USE_PIPELINE}")
print(f"Fine-tuned:   {USE_FINETUNED}")
print(f"Dataset dir:  {DATASET_DIR}")
print(f"LoRA path:    {LORA_PATH if USE_FINETUNED else 'N/A'}")

---
## 📥 Download Dataset (if not attached)

In [ ]:
import os

JSONL_PATH = os.path.join(DATASET_DIR, "kaggle_dataset_qwen_vl.jsonl")

if not os.path.exists(JSONL_PATH) and DATASET_SLUG:
    print(f"📥 Downloading dataset: {DATASET_SLUG}")
    !pip install -q kaggle
    # Uses KAGGLE_USERNAME + KAGGLE_KEY env vars, or ~/.kaggle/kaggle.json, or Kaggle Secrets
    !kaggle datasets download -d {DATASET_SLUG} -p /kaggle/working/ --unzip
    # Re-check candidates after download
    _candidates = [
        f"/kaggle/working/{DATASET_NAME}",
        "/kaggle/working/",
    ]
    for p in _candidates:
        if os.path.exists(os.path.join(p, "kaggle_dataset_qwen_vl.jsonl")):
            DATASET_DIR = p
            break
    JSONL_PATH = os.path.join(DATASET_DIR, "kaggle_dataset_qwen_vl.jsonl")
    print(f"✅ Downloaded to: {DATASET_DIR}")
else:
    print(f"✅ Dataset ready at: {DATASET_DIR}")

assert os.path.exists(JSONL_PATH), f"❌ JSONL not found at {JSONL_PATH}"
print(f"📄 JSONL: {JSONL_PATH}")

---
## 📦 Install Dependencies

In [ ]:
%%capture
!pip install -q --upgrade pip

if USE_PIPELINE == "unsloth":
    !pip install unsloth
    !pip install --no-deps xformers trl peft accelerate bitsandbytes
    !pip install qwen-vl-utils
else:
    !pip install -q --upgrade "transformers>=4.57.0" accelerate
    !pip install -q --upgrade peft bitsandbytes
    SITE = "/usr/local/lib/python3.12/dist-packages"
    !pip uninstall -y -q pillow
    !rm -rf {SITE}/PIL {SITE}/[Pp]illow-*.dist-info {SITE}/pillow.libs
    !pip install -q --no-cache-dir --no-deps "pillow==11.3.0"

print("✅ Dependencies installed. Restart kernel if this is the first run.")

---
## 🧠 Load Model

In [ ]:
import os, torch, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

assert torch.cuda.is_available(), "❌ No GPU! Enable GPU T4 in Kaggle Settings."
print(f"GPU: {torch.cuda.get_device_name(0)} — {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

if USE_PIPELINE == "unsloth":
    from unsloth import FastVisionModel

    model_to_load = LORA_PATH if USE_FINETUNED else UNSLOTH_BASE_MODEL
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=model_to_load,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    FastVisionModel.for_inference(model)
    processor = tokenizer

else:
    from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        OFFICIAL_BASE_MODEL,
        quantization_config=bnb_config,
        device_map={"": 0},
        dtype=torch.float16,
    )

    if USE_FINETUNED:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, LORA_PATH)

    model.eval()
    processor = AutoProcessor.from_pretrained(
        LORA_PATH if USE_FINETUNED else OFFICIAL_BASE_MODEL
    )
    processor.image_processor.size = {"shortest_edge": MIN_PIXELS, "longest_edge": MAX_PIXELS}
    processor.image_processor.min_pixels = MIN_PIXELS
    processor.image_processor.max_pixels = MAX_PIXELS

label = "fine-tuned" if USE_FINETUNED else "base"
print(f"\n✅ Model loaded ({USE_PIPELINE}, {label})")
print(f"   GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

---
## 🔮 Inference Function

In [ ]:
from PIL import Image, ImageOps

SYSTEM_PROMPT = """You are a highly accurate document analysis AI specialized in processing Proof of Delivery (POD) images.
You must analyze the provided POD image and extract specific information.

Classification Rules (in priority order):
1. Physical Paper Damage (torn, ripped, missing sections) -> MANUAL_CHECK_REQUIRED
2. Damage + Short mentioned in remarks -> ISSUE_POD_DAMAGED_AND_SHORT
3. Damage mentioned in remarks -> ISSUE_POD_DAMAGED
4. Shortage mentioned in remarks -> ISSUE_POD_SHORT
5. Seal/Stamp + Signature present -> CLEAN_POD_SEAL_AND_SIGNATURE
6. Seal/Stamp Only -> CLEAN_POD_ONLY_SEAL
7. Signature Only -> CLEAN_POD_ONLY_SIGNATURE
8. Neither Seal nor Signature -> NO_SIGNATURE_NO_STAMP

Output ONLY valid JSON with these exact fields:
- cnNumber: string or null (consignment number)
- hasSignature: boolean
- hasStamp: boolean
- hasHandwriting: boolean
- imageQualityPassed: boolean
- remarksText: string or null
- deliveryDate: "YYYY-MM-DD" or null
- categoryReason: string (explain your classification)
- confidenceScore: float (0.0 to 1.0)(Based on the confidence of seleteced category)
- podCategory: string (one of the categories above)
- limit_exceed: boolean (false for normal deliveries)"""

USER_PROMPT = "Extract all POD fields into JSON."


def load_image(path):
    if path.startswith("file://"):
        path = path[7:]
    with Image.open(path) as img:
        return ImageOps.exif_transpose(img).convert("RGB")


def parse_json(text):
    clean = text.strip()
    for prefix in ["```json", "```"]:
        if clean.startswith(prefix):
            clean = clean[len(prefix):]
    if clean.endswith("```"):
        clean = clean[:-3]
    return json.loads(clean.strip())


def predict_pod(image_input):
    """Run inference on a single POD image (path, URL, or PIL Image)."""

    if isinstance(image_input, str):
        image_content = {"type": "image", "image": image_input}
        pil_image = load_image(image_input)
    else:
        image_content = {"type": "image", "image": image_input}
        pil_image = image_input

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            image_content,
            {"type": "text", "text": USER_PROMPT},
        ]},
    ]

    if USE_PIPELINE == "unsloth":
        from qwen_vl_utils import process_vision_info
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = tokenizer(
            text=[input_text], images=image_inputs, videos=video_inputs,
            padding=True, return_tensors="pt",
        ).to(model.device)
    else:
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text], images=[pil_image],
            return_tensors="pt", padding=True,
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            top_p=0.95,
            do_sample=True,
            repetition_penalty=1.1,
            use_cache=True,
        )

    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    decode_tok = tokenizer if USE_PIPELINE == "unsloth" else processor.tokenizer
    response = decode_tok.batch_decode(generated, skip_special_tokens=True)[0]

    try:
        return parse_json(response)
    except json.JSONDecodeError:
        return {"raw_output": response, "parse_error": True}


print("✅ predict_pod() ready")

---
## 📊 Evaluate on Dataset

In [ ]:
if NUM_EVAL_SAMPLES > 0 and os.path.exists(JSONL_PATH):
    samples = []
    with open(JSONL_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entry = json.loads(line)
            messages = entry.get("messages", [])

            image_path = None
            for msg in messages:
                if msg["role"] == "user" and isinstance(msg["content"], list):
                    for item in msg["content"]:
                        if isinstance(item, dict) and item.get("type") == "image":
                            image_path = item.get("image", "")
                            break
            if not image_path:
                continue

            if image_path.startswith("file://"):
                image_path = image_path[7:]
            abs_path = os.path.join(DATASET_DIR, image_path)
            if not os.path.exists(abs_path):
                continue

            answer = ""
            for msg in messages:
                if msg["role"] == "assistant":
                    answer = msg["content"] if isinstance(msg["content"], str) else "".join(
                        item.get("text", "") for item in msg["content"] if isinstance(item, dict)
                    )
                    break

            samples.append({"image_path": abs_path, "answer": answer})

    print(f"Loaded {len(samples)} samples")

    import random
    random.seed(42)
    random.shuffle(samples)
    eval_samples = samples[:NUM_EVAL_SAMPLES]

    correct = 0
    valid_json = 0

    for i, s in enumerate(eval_samples):
        print(f"\n--- Sample {i+1}/{len(eval_samples)} ---")
        try:
            expected = json.loads(s["answer"])
        except Exception:
            expected = {}

        pred = predict_pod(s["image_path"])

        is_valid = "parse_error" not in pred
        if is_valid:
            valid_json += 1
        cat_match = is_valid and pred.get("podCategory") == expected.get("podCategory")
        if cat_match:
            correct += 1

        print(f"  Expected:  {expected.get('podCategory', 'N/A')}")
        print(f"  Predicted: {pred.get('podCategory', 'PARSE_ERROR')}")
        print(f"  Match: {'✅' if cat_match else '❌'}  |  Valid JSON: {'✅' if is_valid else '❌'}")
        torch.cuda.empty_cache()

    n = max(len(eval_samples), 1)
    print(f"\n{'='*50}")
    print(f"Valid JSON: {valid_json}/{n} ({100*valid_json/n:.0f}%)")
    print(f"Category:   {correct}/{n} ({100*correct/n:.0f}%)")
    print(f"{'='*50}")
else:
    print("⏭️ Skipping dataset eval (NUM_EVAL_SAMPLES=0 or dataset not found)")

---
## 🖼️ Single Image Demo

In [ ]:
import matplotlib.pyplot as plt

if 'eval_samples' in dir() and eval_samples:
    demo_path = eval_samples[0]["image_path"]
elif 'samples' in dir() and samples:
    demo_path = samples[0]["image_path"]
else:
    demo_path = "/kaggle/input/your-image.jpg"

print(f"Running inference on: {demo_path}")
result = predict_pod(demo_path)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

img = load_image(demo_path)
axes[0].imshow(img)
axes[0].set_title("POD Image", fontsize=14)
axes[0].axis("off")

axes[1].text(
    0.05, 0.95, json.dumps(result, indent=2)[:800],
    transform=axes[1].transAxes, fontsize=9,
    va="top", fontfamily="monospace",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
)
label = f"Prediction ({USE_PIPELINE}, {'fine-tuned' if USE_FINETUNED else 'base'})"
axes[1].set_title(label, fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/inference_demo.png", dpi=150)
plt.show()

print("\n📋 Full output:")
print(json.dumps(result, indent=2))